# Tree Lab

In [1]:
# imports and config
import logging
import os
import sys
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

cwd = Path.cwd()
root = cwd if (cwd / "src").exists() else cwd.parents[1]
if str(root) not in sys.path:
    sys.path.append(str(root))

from src.llm.llm import OpenAILLM
from src.preprocessing.reader import Reader
from src.tree_rag.preprocessing import TreePreprocessor
from src.tree_rag.tree import DocumentTree

load_dotenv()

debug = False
logging.basicConfig(
    level=logging.DEBUG if debug else logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("tree_lab")

base_url = os.getenv("BASE_OPEN_AI_URL")
api_key = os.getenv("OPEN_AI_KEY")
client = OpenAI(api_key=api_key, base_url=base_url)
llm = OpenAILLM(client=client, model_name="gpt-5.4-mini")
reader = Reader(logger=logger)
preprocessor = TreePreprocessor(llm=llm, logger=logger)

raw_data_path = root / "data" / "raw_data"
pdf_path = sorted(raw_data_path.glob("*.pdf"))[0]
pdf_path


/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/danilaandreev/Documents/HSE/degree/data/raw_data/4293720391.pdf')

## Preprocessing

In [ ]:
# read pdf
document = reader.read(str(pdf_path))
assert document is not None, f"Failed to read {pdf_path}"

len(document.pages), document.pages[0].number, document.pages[0].text[:500]


In [ ]:
# extract toc
toc = preprocessor.extract_toc(document)

print("has_toc =", toc.has_toc)
print("toc_pages =", toc.toc_pages)
print("entries =", len(toc.entries))
toc.entries


In [ ]:
# resolve page spans
assert toc.has_toc and toc.entries, "TOC was not extracted"
spans = preprocessor.resolve_section_spans(toc.entries, document, toc.toc_pages)
spans


In [ ]:
# build tree
tree = preprocessor.build_tree_from_spans(document, spans)
root_node = tree.get_node(tree.root_id)
print(root_node.preview(depth=None))


In [ ]:
# inspect document description
pprint(tree.document_description)


In [ ]:
# save tree
out_dir = root / "src" / "tree_rag" / "data" / "docs_tree"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{pdf_path.stem}_tree.json"
tree.save(out_path)
out_path


## Viewing


In [12]:
out_dir = root / "src" / "tree_rag" / "data" / "docs_tree"
out_path = out_dir / f"{pdf_path.stem}_tree.json"
loaded_tree = DocumentTree.load(out_path)

In [13]:
pprint(loaded_tree.document_description)

('Документ содержит требования к строительству, монтажу, испытаниям и приемке '
 'трубопроводов и сооружений систем водоснабжения и канализации, включая '
 'земляные работы, соединения труб, переходы через препятствия, водозаборные и '
 'емкостные сооружения. Также рассматриваются промывка, дезинфекция и '
 'особенности работ в специальных природно-климатических условиях.')


In [15]:
# load saved tree and open context for llm
labels = {
    5: ['*', '!'],
    9: ['!'],
    11: ['*'],
}
print(loaded_tree.llm_view(labels=labels))

- 4293720391
  - [1] Область применения
  - [2] Нормативные ссылки
  - [3] Термины и определения
  - [4] Общие положения
  - [5][*][!] Земляные работы
  - Монтаж трубопроводов
    - Требования к монтажу трубопроводов
      - [6] Общие требования к монтажу
      - Сварка и сборка стальных труб
        - [7] Общие требования к сварке и сборке
        - [8] Условия сварки и требования к выполнению работ
        - Контроль качества сварных соединений
          - [9][!] Контроль качества сварных швов
          - [10] Исправление дефектов сварных швов
    - Соединения стеклокомпозитных труб
      - [11][*] Общие требования и виды соединений
      - [12] Муфтовые и раструбные соединения
      - Монтаж муфты и стыковка труб
        - [13] Надевание муфты на трубу
        - [14] Подготовка соединения и стыковка труб
        - [15] Требования к соединению и смазке
      - [16] Фланцевые и механические соединения
      - [17] Клеевые и ламинированные соединения
    - [18] Требования к стыкам труб

In [7]:
loaded_tree.max_leaf_idx()

41

In [4]:
# inspect text by leaf idx
leaf_idx = 13
print(loaded_tree.get_text_by_idx(leaf_idx))

Path: 4293720391.pdf / Монтаж трубопроводов / Соединения стеклокомпозитных труб / Монтаж муфты и стыковка труб / Надевание муфты на трубу
Content:
6.3.6 Стеклокомпозитные трубы могут поставляться с предварительно надетой муфтой на один конец трубы, а также раздельно. В случае раздельной поставки муфта надевается на трубу перед мон­ тажом. На трубу надевается строп или стальной хомут на расстоянии от 1 до 2 м от конца трубы, муфта вывешивается над землей на высоте не менее 100 мм и надевается на гладкий конец трубы, как показано на рисунке 6. С помощью деревянного бруска и двух ручных домкратов или ручных (рычажных) талей муфта натягивается до стопорного кольца или контрольной отметки, нанесенной на трубе. 1 — брус 50 *100; 2 — муфта; 3 — ручная (рычажная) таль; 4 — строп или канат; 5 — стальной хомут Рисунок 6 — Схема надевания муфты на трубу
